# 06 - Clustering experiments

Objetivo: comparar algoritmos y cantidades de grupos sin asumir una solucion unica.

In [ ]:
import sys
from pathlib import Path
import polars as pl
sys.path.insert(0, str(Path.cwd().parent))
from analysis.pipeline import load_transactions, build_customer_features, transform_features, run_clustering, evaluate_labels

ROOT = Path.cwd().parent if Path.cwd().name == "analysis" else Path.cwd()
features = build_customer_features(load_transactions(ROOT / "transacciones_sinteticas.csv"))
matrix, columns = transform_features(features, scaler="robust")
experiments = []
for algorithm in ("kmeans", "minibatch_kmeans", "gmm", "agglomerative", "dbscan", "hdbscan"):
    for k in (3, 4, 5, 6):
        try:
            labels = run_clustering(matrix, algorithm, n_clusters=k)
            experiments.append({"algorithm": algorithm, "n_clusters_parameter": k, **evaluate_labels(matrix, labels)})
        except (ValueError, ImportError) as error:
            print(f"{algorithm}: omitido ({error})")
results = pl.DataFrame(experiments)
results.write_csv(ROOT / "results" / "metrics" / "clustering_experiments.csv")
results.sort("silhouette", descending=True).head(15)

## Decision
La configuracion seleccionada no se decide solo por silhouette: se consideran ruido, tamanos, estabilidad e interpretabilidad en las etapas siguientes.